<a href="https://colab.research.google.com/github/filfish2007-cav/Hospital_LLM/blob/main/DataCleaning_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installing pandas and matplotlib


In [2]:
import pandas as pd
import matplotlib as plt

### Exploratory Data Analysis: Dataset Overview
Inspect the structure, columns, data types, and first few rows of each of the 6 DataFrames.

In [3]:
# =========================
# 1. LOAD DATA
# =========================

base_url = "https://raw.githubusercontent.com/filfish2007-cav/Hospital_LLM/main/data/hospital_csvs/"

files = [
    "patients.csv",
    "encounters.csv",
    "conditions.csv",
    "medications.csv",
    "observations.csv",
    "procedures.csv"
]

data = {
    file.replace(".csv", ""): pd.read_csv(base_url + file)
    for file in files
}


# =========================
# 2. BASIC INSPECTION
# =========================

for name, df in data.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {name.upper()}")
    print("=" * 80)

    # 1. Number of rows / columns
    print("\n1. Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. Column names
    print("\n2. Column names")
    print(df.columns.tolist())

    # 3. Data types
    print("\n3. Data types")
    print(df.dtypes)

    # 4. Missing values
    print("\n4. Missing values")
    missing = df.isnull().sum()
    missing_percentage = (df.isnull().mean() * 100).round(2)

    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing_percentage
    })

    print(missing_report[missing_report["missing_count"] > 0])

    if missing.sum() == 0:
        print("No missing values.")

    # 5. Duplicated rows
    print("\n5. Duplicated rows")
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")

    # 6. Unique IDs
    print("\n6. Potential ID columns")

    id_columns = [
        column for column in df.columns
        if column.lower() == "id"
        or column.lower().endswith("_id")
        or column.lower() == "patient"
        or column.lower() == "encounter"
    ]

    if id_columns:
        for column in id_columns:
            print(
                f"{column}: "
                f"{df[column].nunique()} unique / "
                f"{len(df)} total"
            )
        else:
            print("No obvious ID columns found.")

    # 7. Suspicious numeric values
    print("\n7. Numeric value overview")

    numeric_columns = df.select_dtypes(include="number").columns

    if len(numeric_columns) > 0:
        print(df[numeric_columns].describe().T)
    else:
        print("No numeric columns.")

    # 8. Date ranges
    print("\n8. Date columns / ranges")

    date_columns = []

    for column in df.columns:
        if (
            "date" in column.lower()
            or "time" in column.lower()
            or "birth" in column.lower()
            or "death" in column.lower()
        ):
            date_columns.append(column)

    if date_columns:
        for column in date_columns:

            dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            print(
                f"{column}: "
                f"{dates.min()} → {dates.max()}"
            )
    else:
        print("No obvious date columns found.")

    # 9. Categorical values
    print("\n9. Categorical columns")

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        unique_values = df[column].dropna().unique()

        print(f"\n{column}")
        print(f"Unique values: {len(unique_values)}")

        # Print values only when there aren't too many
        if len(unique_values) <= 30:
            print(unique_values)
        else:
            print("Too many unique values to display.")


# =========================
# 3. RELATIONSHIPS BETWEEN TABLES
# =========================

print("\n" + "=" * 80)
print("10. RELATIONSHIPS BETWEEN TABLES")
print("=" * 80)


# Collect columns from every table
all_columns = {}

for table_name, df in data.items():
    for column in df.columns:
        all_columns.setdefault(column, []).append(table_name)


print("\nColumns appearing in multiple tables:")

for column, tables in all_columns.items():

    if len(tables) > 1:
        print(
            f"{column}: "
            f"{', '.join(tables)}"
        )


# =========================
# 4. FOREIGN KEY CHECKS
# =========================

print("\n" + "=" * 80)
print("FOREIGN KEY / ID INTEGRITY CHECKS")
print("=" * 80)


# Common relationships expected in Synthea
relationships = [
    ("encounters", "PATIENT", "patients", "Id"),
    ("conditions", "PATIENT", "patients", "Id"),
    ("conditions", "ENCOUNTER", "encounters", "Id"),
    ("medications", "PATIENT", "patients", "Id"),
    ("medications", "ENCOUNTER", "encounters", "Id"),
    ("observations", "PATIENT", "patients", "Id"),
    ("observations", "ENCOUNTER", "encounters", "Id"),
    ("procedures", "PATIENT", "patients", "Id"),
    ("procedures", "ENCOUNTER", "encounters", "Id"),
]


for child_table, child_column, parent_table, parent_column in relationships:

    if (
        child_table in data
        and parent_table in data
        and child_column in data[child_table].columns
        and parent_column in data[parent_table].columns
    ):

        child_ids = set(
            data[child_table][child_column]
            .dropna()
            .unique()
        )

        parent_ids = set(
            data[parent_table][parent_column]
            .dropna()
            .unique()
        )

        orphan_ids = child_ids - parent_ids

        print(
            f"\n{child_table}.{child_column}"
            f" → "
            f"{parent_table}.{parent_column}"
        )

        print(f"Referenced IDs: {len(child_ids)}")
        print(f"Missing parent IDs: {len(orphan_ids)}")

        if orphan_ids:
            print("WARNING: orphan IDs detected!")
        else:
            print("OK: all references exist.")


TABLE: PATIENTS

1. Shape
Rows: 120
Columns: 28

2. Column names
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']

3. Data types
Id                      object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
MIDDLE                  object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                  object
BIRTHPLACE              object
ADDRESS                 object
CITY                    object
STATE                   object


/tmp/ipykernel_3816/1251747611.py:113: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


           missing_count  missing_percentage
ENCOUNTER           3189                2.71
CATEGORY            3189                2.71
UNITS              31848               27.10

5. Duplicated rows
Duplicate rows: 22

6. Potential ID columns
PATIENT: 120 unique / 117524 total
ENCOUNTER: 4018 unique / 117524 total
No obvious ID columns found.

7. Numeric value overview
No numeric columns.

8. Date columns / ranges
DATE: 1967-07-01 01:30:03+00:00 → 2026-09-23 13:15:05+00:00

9. Categorical columns

DATE
Unique values: 10929
Too many unique values to display.

PATIENT
Unique values: 120
Too many unique values to display.

ENCOUNTER
Unique values: 4018
Too many unique values to display.

CATEGORY
Unique values: 8
['vital-signs' 'laboratory' 'social-history' 'survey' 'exam' 'procedure'
 'therapy' 'imaging']

CODE
Unique values: 241
Too many unique values to display.

DESCRIPTION
Unique values: 242
Too many unique values to display.

VALUE
Unique values: 3583
Too many unique values to disp

### Exploring duplicates (22 rows) in observation column

In [4]:
duplicates = data["observations"][
    data["observations"].duplicated(keep=False)
]

display(duplicates)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
5622,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5623,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,17.4,mg/dL,numeric
5625,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric
5626,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5627,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6298-4,Potassium [Moles/volume] in Blood,3.8,mmol/L,numeric
5628,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5629,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5646,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5647,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,17.4,mg/dL,numeric
5649,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric


In [5]:
print(f"Total rows in the duplicates DataFrame (all occurrences, keep=False): {len(duplicates)}")
print(f"Redundant duplicate rows (keep='first'): {data['observations'].duplicated(keep='first').sum()}")

Total rows in the duplicates DataFrame (all occurrences, keep=False): 44
Redundant duplicate rows (keep='first'): 22


In [6]:
# Sort duplicates to place identical pairs next to each other
sorted_duplicates = duplicates.sort_values(by=["PATIENT", "DATE", "CODE"])
display(sorted_duplicates)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
5629,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5653,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5628,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5652,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5622,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5646,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5626,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5650,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5625,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric
5649,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric


In [7]:
# 1. Check if 'observations' has a primary key ID used in other tables
# Usually observations don't have an ID column in this dataset, let's verify.
print("Columns in observations:", data["observations"].columns.tolist())

# 2. Check if removing duplicates from observations alters unique patient/encounter values
original_patients = data["observations"]["PATIENT"].nunique()
original_encounters = data["observations"]["ENCOUNTER"].nunique()

clean_observations = data["observations"].drop_duplicates()

clean_patients = clean_observations["PATIENT"].nunique()
clean_encounters = clean_observations["ENCOUNTER"].nunique()

print(f"Patients before: {original_patients} | After: {clean_patients}")
print(f"Encounters before: {original_encounters} | After: {clean_encounters}")

Columns in observations: ['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION', 'VALUE', 'UNITS', 'TYPE']
Patients before: 120 | After: 120
Encounters before: 4018 | After: 4018


### Deleting duplicates since it does not affect other tables

In [8]:
# 1. Create a safe standalone backup variable outside the data dictionary
observations_backup = data["observations"].copy()
if "observations_backup" in data:
    del data["observations_backup"]

# 2. Drop duplicates in place from the active observations table
original_row_count = len(data["observations"])
data["observations"] = data["observations"].drop_duplicates()
new_row_count = len(data["observations"])

print("Backup created as standalone variable 'observations_backup' (removed from 'data' dictionary).")
print(f"Active table deduplicated: {original_row_count} rows -> {new_row_count} rows (removed {original_row_count - new_row_count} duplicate rows).")

Backup created as standalone variable 'observations_backup' (removed from 'data' dictionary).
Active table deduplicated: 117524 rows -> 117502 rows (removed 22 duplicate rows).


In [9]:
# 1. Ensure the backup key is completely removed from the data dictionary
if "observations_backup" in data:
    del data["observations_backup"]

# 2. Verify that the standalone backup variable still exists and is safe in memory
try:
    print(f"✓ Standalone 'observations_backup' is available!")
    print(f"  Shape: {observations_backup.shape}")
    print(f"  Active 'data' dictionary keys: {list(data.keys())}")
except NameError:
    print("❌ Standalone backup was not found. Re-creating it now...")
    observations_backup = data["observations"].copy()
    print("✓ Standalone 'observations_backup' created successfully!")

✓ Standalone 'observations_backup' is available!
  Shape: (117524, 9)
  Active 'data' dictionary keys: ['patients', 'encounters', 'conditions', 'medications', 'observations', 'procedures']


In [10]:
for name, df in data.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {name.upper()}")
    print("=" * 80)

    # 1. Number of rows / columns
    print("\n1. Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. Column names
    print("\n2. Column names")
    print(df.columns.tolist())

    # 3. Data types
    print("\n3. Data types")
    print(df.dtypes)

    # 4. Missing values
    print("\n4. Missing values")
    missing = df.isnull().sum()
    missing_percentage = (df.isnull().mean() * 100).round(2)

    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing_percentage
    })

    print(missing_report[missing_report["missing_count"] > 0])

    if missing.sum() == 0:
        print("No missing values.")

    # 5. Duplicated rows
    print("\n5. Duplicated rows")
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")

    # 6. Unique IDs
    print("\n6. Potential ID columns")

    id_columns = [
        column for column in df.columns
        if column.lower() == "id"
        or column.lower().endswith("_id")
        or column.lower() == "patient"
        or column.lower() == "encounter"
    ]

    if id_columns:
        for column in id_columns:
            print(
                f"{column}: "
                f"{df[column].nunique()} unique / "
                f"{len(df)} total"
            )
        else:
            print("No obvious ID columns found.")

    # 7. Suspicious numeric values
    print("\n7. Numeric value overview")

    numeric_columns = df.select_dtypes(include="number").columns

    if len(numeric_columns) > 0:
        print(df[numeric_columns].describe().T)
    else:
        print("No numeric columns.")

    # 8. Date ranges
    print("\n8. Date columns / ranges")

    date_columns = []

    for column in df.columns:
        if (
            "date" in column.lower()
            or "time" in column.lower()
            or "birth" in column.lower()
            or "death" in column.lower()
        ):
            date_columns.append(column)

    if date_columns:
        for column in date_columns:

            dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            print(
                f"{column}: "
                f"{dates.min()} → {dates.max()}"
            )
    else:
        print("No obvious date columns found.")

    # 9. Categorical values
    print("\n9. Categorical columns")

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        unique_values = df[column].dropna().unique()

        print(f"\n{column}")
        print(f"Unique values: {len(unique_values)}")

        # Print values only when there aren't too many
        if len(unique_values) <= 30:
            print(unique_values)
        else:
            print("Too many unique values to display.")


# =========================
# 3. RELATIONSHIPS BETWEEN TABLES
# =========================

print("\n" + "=" * 80)
print("10. RELATIONSHIPS BETWEEN TABLES")
print("=" * 80)


# Collect columns from every table
all_columns = {}

for table_name, df in data.items():
    for column in df.columns:
        all_columns.setdefault(column, []).append(table_name)


print("\nColumns appearing in multiple tables:")

for column, tables in all_columns.items():

    if len(tables) > 1:
        print(
            f"{column}: "
            f"{', '.join(tables)}"
        )


# =========================
# 4. FOREIGN KEY CHECKS
# =========================

print("\n" + "=" * 80)
print("FOREIGN KEY / ID INTEGRITY CHECKS")
print("=" * 80)


# Common relationships expected in Synthea
relationships = [
    ("encounters", "PATIENT", "patients", "Id"),
    ("conditions", "PATIENT", "patients", "Id"),
    ("conditions", "ENCOUNTER", "encounters", "Id"),
    ("medications", "PATIENT", "patients", "Id"),
    ("medications", "ENCOUNTER", "encounters", "Id"),
    ("observations", "PATIENT", "patients", "Id"),
    ("observations", "ENCOUNTER", "encounters", "Id"),
    ("procedures", "PATIENT", "patients", "Id"),
    ("procedures", "ENCOUNTER", "encounters", "Id"),
]


for child_table, child_column, parent_table, parent_column in relationships:

    if (
        child_table in data
        and parent_table in data
        and child_column in data[child_table].columns
        and parent_column in data[parent_table].columns
    ):

        child_ids = set(
            data[child_table][child_column]
            .dropna()
            .unique()
        )

        parent_ids = set(
            data[parent_table][parent_column]
            .dropna()
            .unique()
        )

        orphan_ids = child_ids - parent_ids

        print(
            f"\n{child_table}.{child_column}"
            f" → "
            f"{parent_table}.{parent_column}"
        )

        print(f"Referenced IDs: {len(child_ids)}")
        print(f"Missing parent IDs: {len(orphan_ids)}")

        if orphan_ids:
            print("WARNING: orphan IDs detected!")
        else:
            print("OK: all references exist.")


TABLE: PATIENTS

1. Shape
Rows: 120
Columns: 28

2. Column names
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']

3. Data types
Id                      object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
MIDDLE                  object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                  object
BIRTHPLACE              object
ADDRESS                 object
CITY                    object
STATE                   object


/tmp/ipykernel_3816/677355841.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


           missing_count  missing_percentage
ENCOUNTER           3189                2.71
CATEGORY            3189                2.71
UNITS              31848               27.10

5. Duplicated rows
Duplicate rows: 0

6. Potential ID columns
PATIENT: 120 unique / 117502 total
ENCOUNTER: 4018 unique / 117502 total
No obvious ID columns found.

7. Numeric value overview
No numeric columns.

8. Date columns / ranges
DATE: 1967-07-01 01:30:03+00:00 → 2026-09-23 13:15:05+00:00

9. Categorical columns

DATE
Unique values: 10929
Too many unique values to display.

PATIENT
Unique values: 120
Too many unique values to display.

ENCOUNTER
Unique values: 4018
Too many unique values to display.

CATEGORY
Unique values: 8
['vital-signs' 'laboratory' 'social-history' 'survey' 'exam' 'procedure'
 'therapy' 'imaging']

CODE
Unique values: 241
Too many unique values to display.

DESCRIPTION
Unique values: 242
Too many unique values to display.

VALUE
Unique values: 3583
Too many unique values to displ

### Standardising column names

In [11]:
# Inspect the exact column names of all loaded tables to identify naming discrepancies
for name, df in data.items():
    columns = df.columns.tolist()
    # Check for any leading/trailing spaces in column names
    spaced_columns = [col for col in columns if col != col.strip()]

    print(f"Table: {name.upper()}")
    print(f"  Columns: {columns}")
    if spaced_columns:
        print(f"  ⚠️ WARNING: Columns with hidden spaces: {spaced_columns}")
    else:
        print("  ✓ No hidden spaces in column names.")
    print("-" * 50)

Table: PATIENTS
  Columns: ['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: ENCOUNTERS
  Columns: ['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: CONDITIONS
  Columns: ['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: MEDICATIONS
  Columns: ['START', 'STOP', 'PATIENT', 'PAYER

In [12]:
import re

def to_snake_case(name):
    # Handle common camelCase or PascalCase split points first
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    # Replace potential double underscores and clean up
    return re.sub('_+', '_', s2)

# Standardize column headers for each active table in 'data'
for table_name, df in data.items():

    # Apply snake_case transformation
    new_columns = [to_snake_case(col) for col in df.columns]

    # Specifically handle standard ID columns we want as simple 'id'
    new_columns = ['id' if col == 'id' else col for col in new_columns]

    df.columns = new_columns
    print(f"Table '{table_name}' updated columns:")
    print(df.columns.tolist())
    print("-" * 50)

Table 'patients' updated columns:
['id', 'birthdate', 'deathdate', 'ssn', 'drivers', 'passport', 'prefix', 'first', 'middle', 'last', 'suffix', 'maiden', 'marital', 'race', 'ethnicity', 'gender', 'birthplace', 'address', 'city', 'state', 'county', 'fips', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthcare_coverage', 'income']
--------------------------------------------------
Table 'encounters' updated columns:
['id', 'start', 'stop', 'patient', 'organization', 'provider', 'payer', 'encounterclass', 'code', 'description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reasoncode', 'reasondescription']
--------------------------------------------------
Table 'conditions' updated columns:
['start', 'stop', 'patient', 'encounter', 'system', 'code', 'description']
--------------------------------------------------
Table 'medications' updated columns:
['start', 'stop', 'patient', 'payer', 'encounter', 'code', 'description', 'base_cost', 'payer_coverage', 'dispenses', 'to

### patiens table - checking ids

In [13]:
import re

patients_df = data['patients']

# 1. Check for Nulls
null_count = patients_df['id'].isnull().sum()
print(f"Null IDs count: {null_count}")

# 2. Check for Whitespace and Clean in-place
patients_df['id'] = patients_df['id'].astype(str).str.strip()
print("Strip complete. Whitespace has been removed from 'id' column.")

# 3. Check for uniqueness
unique_ids = patients_df['id'].nunique()
total_rows = len(patients_df)
print(f"Unique IDs: {unique_ids} | Total Rows: {total_rows}")
if unique_ids == total_rows:
    print("✓ All patient IDs are completely unique.")
else:
    print("⚠️ WARNING: Duplicate patient IDs detected!")

# 4. Check UUID format (8-4-4-4-12 hex characters)
uuid_pattern = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.IGNORECASE)
valid_uuid_mask = patients_df['id'].apply(lambda x: bool(uuid_pattern.match(x)))
invalid_uuids = patients_df[~valid_uuid_mask]['id']

print(f"Valid UUID format count: {valid_uuid_mask.sum()} / {total_rows}")
if not invalid_uuids.empty:
    print(f"⚠️ WARNING: Found {len(invalid_uuids)} IDs with an invalid format:")
    print(invalid_uuids.head())
else:
    print("✓ All patient IDs conform to a standard UUID format.")

Null IDs count: 0
Strip complete. Whitespace has been removed from 'id' column.
Unique IDs: 120 | Total Rows: 120
✓ All patient IDs are completely unique.
Valid UUID format count: 120 / 120
✓ All patient IDs conform to a standard UUID format.


### Patients - Checking Dates

In [15]:
import pandas as pd

# Safely parse dates to datetime
patients_df = data['patients']
patients_df['birthdate'] = pd.to_datetime(patients_df['birthdate'], errors='coerce')
patients_df['deathdate'] = pd.to_datetime(patients_df['deathdate'], errors='coerce')

print("1. Basic Null Checks:")
print(f"Parsed birthdates missing: {patients_df['birthdate'].isna().sum()}")
print(f"Parsed deathdates missing: {patients_df['deathdate'].isna().sum()} (out of {len(patients_df)})")

1. Basic Null Checks:
Parsed birthdates missing: 0
Parsed deathdates missing: 100 (out of 120)


In [16]:
# 2. logical check: deathdate >= birthdate
invalid_death_order = patients_df[patients_df['deathdate'] < patients_df['birthdate']]
print(f"\n2. Logical check: Patients with deathdate < birthdate: {len(invalid_death_order)}")
if not invalid_death_order.empty:
    display(invalid_death_order[['id', 'birthdate', 'deathdate']])


2. Logical check: Patients with deathdate < birthdate: 0


In [17]:
# 3. Check for future dates relative to current logic (e.g. up to 2026 as observed in date ranges)
reference_date = pd.to_datetime('2026-12-31')
future_births = patients_df[patients_df['birthdate'] > reference_date]
print(f"\n3. Future birthdates check (after 2026-12-31): {len(future_births)}")

# Calculate approximate age
patients_df['age_at_reference'] = (reference_date - patients_df['birthdate']).dt.days / 365.25
extremely_old = patients_df[patients_df['age_at_reference'] > 110]
print(f"Patients older than 110 years (relative to 2026): {len(extremely_old)}")
if not extremely_old.empty:
    display(extremely_old[['id', 'birthdate', 'age_at_reference']])


3. Future birthdates check (after 2026-12-31): 0
Patients older than 110 years (relative to 2026): 0


In [22]:
# 4. Cross-table consistency: Events before birthdate or after deathdate
# Parse dates in encounters and observations for precise comparisons
encounters_df = data['encounters'].copy()
encounters_df['start'] = pd.to_datetime(encounters_df['start'], errors='coerce').dt.tz_localize(None)
encounters_df['stop'] = pd.to_datetime(encounters_df['stop'], errors='coerce').dt.tz_localize(None)

observations_df = data['observations'].copy()
observations_df['date'] = pd.to_datetime(observations_df['date'], errors='coerce').dt.tz_localize(None)

# Align patients_df timezone
p_temp = patients_df.copy()
p_temp['birthdate'] = p_temp['birthdate'].dt.tz_localize(None)
p_temp['deathdate'] = p_temp['deathdate'].dt.tz_localize(None)

# Merge patient boundaries with clinical events
enc_check = encounters_df.merge(p_temp[['id', 'birthdate', 'deathdate']], left_on='patient', right_on='id')
obs_check = observations_df.merge(p_temp[['id', 'birthdate', 'deathdate']], left_on='patient', right_on='id')

# Check encounters/observations occurring before patient birth
enc_before_birth = enc_check[enc_check['start'] < enc_check['birthdate']]
obs_before_birth = obs_check[obs_check['date'] < obs_check['birthdate']]

print(f"\n4. Timeline anomalies (Before Birth):")
print(f"Encounters starting before birth: {len(enc_before_birth)}")
print(f"Observations recorded before birth: {len(obs_before_birth)}")

if not enc_before_birth.empty:
    print("Sample of encounters before birth:")
    display(enc_before_birth[['patient', 'birthdate', 'id_x', 'start', 'description']].head())

# Check encounters/observations occurring after patient death
enc_after_death = enc_check[(enc_check['deathdate'].notna()) & (enc_check['stop'] > enc_check['deathdate'])]
obs_after_death = obs_check[(obs_check['deathdate'].notna()) & (obs_check['date'] > obs_check['deathdate'])]

print(f"\n5. Timeline anomalies (After Death):")
print(f"Encounters finishing after death: {len(enc_after_death)}")
print(f"Observations recorded after death: {len(obs_after_death)}")

if not enc_after_death.empty:
    print("Sample of encounters after death:")
    display(enc_after_death[['patient', 'deathdate', 'id_x', 'stop', 'description']].head())


4. Timeline anomalies (Before Birth):
Encounters starting before birth: 4
Observations recorded before birth: 97
Sample of encounters before birth:


,patient,birthdate,id_x,start,description
0,f19b9cae-0ea7-2d51-f2c5-946d448992a5,2020-01-15,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,2020-01-14 22:59:24,Well child visit (procedure)
143,13a674ff-e2f7-06ad-461b-e4087dcedbd4,2021-04-05,13a674ff-e2f7-06ad-ed74-af311c8b37a6,2021-04-04 22:52:49,Well child visit (procedure)
1088,f4330273-be0c-2edf-2b4e-78bd87b98ee9,2020-04-03,f4330273-be0c-2edf-8de7-09376c61b04c,2020-04-02 21:37:39,Well child visit (procedure)
2258,079614f4-526c-c3eb-a0ef-3ad3d1191236,2005-09-19,079614f4-526c-c3eb-1acd-f16e0bf67397,2005-09-18 22:42:40,Well child visit (procedure)



5. Timeline anomalies (After Death):
Encounters finishing after death: 31
Observations recorded after death: 79
Sample of encounters after death:


,patient,deathdate,id_x,stop,description
324,eec13363-c6e9-a26d-3750-814976e54a24,2013-04-06,eec13363-c6e9-a26d-528c-ebe26eb1e054,2013-04-14 07:08:34,Death Certification
540,5e816607-ff3f-8bc7-7f53-8f272a88f9a4,1970-07-24,5e816607-ff3f-8bc7-0a8f-1ba65661779d,1970-07-24 07:04:32,Admission to hospice (procedure)
555,5e816607-ff3f-8bc7-7f53-8f272a88f9a4,1970-07-24,5e816607-ff3f-8bc7-e4bd-8c5d1f9760eb,1970-07-25 01:45:03,Death Certification
1056,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,381d4a6a-695b-2f73-556c-0c24083525b6,2025-09-02 03:12:27,Admission to intensive care unit (procedure)
1057,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,381d4a6a-695b-2f73-7833-e7f8714e0ef8,2025-09-13 01:45:03,Death Certification


### Fixing anomalies(Clinical Events Recorded BEFORE Birth, Clinical Events Recorded AFTER Death)

In [23]:
tol = pd.Timedelta(days=1)

def to_naive_utc(s):
    return pd.to_datetime(s, utc=True, errors='coerce').dt.tz_localize(None)

p = data['patients'][['id', 'birthdate', 'deathdate']].copy()
p['birthdate'] = pd.to_datetime(p['birthdate']).dt.tz_localize(None)
p['deathdate'] = pd.to_datetime(p['deathdate']).dt.tz_localize(None)

enc = data['encounters'].copy()
enc['start'] = to_naive_utc(enc['start'])
enc = enc.merge(p, left_on='patient', right_on='id', suffixes=('', '_p'))

obs = data['observations'].copy()
obs['date'] = to_naive_utc(obs['date'])
obs = obs.merge(p, left_on='patient', right_on='id', suffixes=('', '_p'))

enc_before = enc[enc['start'] < enc['birthdate'] - tol]
enc_after  = enc[enc['deathdate'].notna() & (enc['start'] > enc['deathdate'] + tol)]
obs_before = obs[obs['date'] < obs['birthdate'] - tol]
obs_after  = obs[obs['deathdate'].notna() & (obs['date'] > obs['deathdate'] + tol)]





0 18 0 21


In [25]:
print("=== TIMELINE ANOMALIES AUDIT (with 1-Day Tolerance) ===")
print(f"Encounters starting > 1 day before birth: {len(enc_before)}")
print(f"Encounters starting > 1 day after death:  {len(enc_after)}")
print(f"Observations recorded > 1 day before birth: {len(obs_before)}")
print(f"Observations recorded > 1 day after death:  {len(obs_after)}")


=== TIMELINE ANOMALIES AUDIT (with 1-Day Tolerance) ===
Encounters starting > 1 day before birth: 0
Encounters starting > 1 day after death:  18
Observations recorded > 1 day before birth: 0
Observations recorded > 1 day after death:  21


### Timeline anomalies: events before birth

4 encounters and 97 observations appeared before the birthdate. All are newborn "Well child visit" encounters starting a few hours before midnight UTC on the day before the birthdate. The cause is date-only birthdates compared against UTC timestamps, not a data error.

With a 1-day tolerance, 0 remain. **No rows removed or modified.**

In [26]:
# What kinds of encounters are they?
print(enc_after['description'].value_counts())

# How far after death, and which patients?
enc_after = enc_after.assign(days_after=(enc_after['start'] - enc_after['deathdate']).dt.days)
display(enc_after[['patient', 'deathdate', 'start', 'days_after', 'description']]
        .sort_values('days_after', ascending=False))

# Are the observations tied to those same encounters?
print(obs_after['encounter'].isin(enc_after['id']).mean())
print(obs_after['description'].value_counts().head(10))

description
Death Certification    18
Name: count, dtype: int64


,patient,deathdate,start,days_after,description
4819,7d4e9f28-6f5a-edb5-2778-cecd6efc4e9a,2021-09-23,2021-10-07 03:16:31,14,Death Certification
4404,1937dc8b-92c6-408d-ed95-ed2c384446a2,2010-05-22,2010-06-03 05:01:28,12,Death Certification
1057,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,2025-09-13 01:30:03,11,Death Certification
1472,9ae2ef41-6004-7e78-7001-1fe4da06ec90,2026-06-18,2026-06-28 19:28:17,10,Death Certification
3736,6f353dad-c5cc-3cdd-87af-fe7eca4dc35b,2015-05-12,2015-05-21 05:01:28,9,Death Certification
3929,f45ba14c-644d-c1fb-1239-fee274e0abb5,1997-10-07,1997-10-16 05:01:28,9,Death Certification
3576,64cc597f-de2d-b26f-a303-8b8beda99395,2022-08-06,2022-08-14 00:05:05,8,Death Certification
324,eec13363-c6e9-a26d-3750-814976e54a24,2013-04-06,2013-04-14 06:53:34,8,Death Certification
6135,a6b49352-759a-1fa6-d169-3f8dd7471305,2000-02-03,2000-02-08 22:08:49,5,Death Certification
5146,5dfaf183-aff9-c886-72ef-492c34861077,2006-11-02,2006-11-07 22:08:49,5,Death Certification


0.8571428571428571
description
Cause of Death [US Standard Certificate of Death]    18
QALY                                                  1
DALY                                                  1
QOLS                                                  1
Name: count, dtype: int64


In [27]:
# Flag death-related administrative events instead of removing them
death_admin_ids = enc_after['id']
data['encounters']['is_death_certification'] = (
    data['encounters']['description'].str.contains('Death Certification', na=False)
)
data['observations']['is_cause_of_death'] = (
    data['observations']['description'].str.contains('Cause of Death', na=False)
)

### Timeline anomalies: events after death

18 encounters and 21 observations fall more than 1 day after the death date. All are expected administrative records:
- **Encounters (18):** all "Death Certification", filed 1-14 days after death.
- **Observations (21):** 18 "Cause of Death" (linked to those encounters) + 3 summary metrics (QALY, DALY, QOLS).

No clinical care occurs after death. **No rows removed**; added flags `is_death_certification` and `is_cause_of_death`.

In [29]:
p = data['patients']
cols = ['gender', 'marital', 'race', 'ethnicity', 'prefix', 'suffix']

for c in cols:
    s = p[c]
    v = s.dropna().astype(str)
    print(f"\n{c}  (missing: {s.isna().sum()})")
    print(s.value_counts(dropna=False).to_dict())
    print(f"  whitespace issues: {(v != v.str.strip()).sum()} | "
          f"empty strings: {(v.str.strip() == '').sum()} | "
          f"case variants: {v.str.lower().nunique() != v.nunique()}")

# Consistency: prefix vs gender, missing marital/prefix vs age
print(pd.crosstab(p['prefix'].fillna('NA'), p['gender']))
age = (pd.Timestamp('2026-09-30') - pd.to_datetime(p['birthdate'])).dt.days / 365.25
print(age.groupby(p['marital'].isna()).median())


gender  (missing: 0)
{'F': 63, 'M': 57}
  whitespace issues: 0 | empty strings: 0 | case variants: False

marital  (missing: 39)
{'M': 47, nan: 39, 'S': 19, 'D': 12, 'W': 3}
  whitespace issues: 0 | empty strings: 0 | case variants: False

race  (missing: 0)
{'white': 104, 'asian': 6, 'black': 5, 'hawaiian': 3, 'other': 2}
  whitespace issues: 0 | empty strings: 0 | case variants: False

ethnicity  (missing: 0)
{'nonhispanic': 105, 'hispanic': 15}
  whitespace issues: 0 | empty strings: 0 | case variants: False

prefix  (missing: 26)
{'Mr.': 43, 'Mrs.': 37, nan: 26, 'Ms.': 14}
  whitespace issues: 0 | empty strings: 0 | case variants: False

suffix  (missing: 119)
{nan: 119, 'JD': 1}
  whitespace issues: 0 | empty strings: 0 | case variants: False
gender   F   M
prefix        
Mr.      0  43
Mrs.    37   0
Ms.     14   0
NA      12  14
marital
False    61.237509
True     15.813826
Name: birthdate, dtype: float64


### Patients: demographic columns

`gender`, `marital`, `race`, `ethnicity`, `prefix`, `suffix` all contain only valid categories, with no whitespace, case variants or empty strings. Prefix matches gender in all rows.

Missing `marital` (39) and `prefix` (26) are mostly minors (median age ~16 and ~12), so they are left as NaN. `suffix` is almost empty (1 value: "JD"). **No changes made.**

In [32]:
import pandas as pd
import numpy as np

# Reload ZIP and FIPS as strings. pandas reads ZIP as int, which drops leading zeros
# (01906 becomes 1906) and turns the '00000' placeholder into 0.
geo = pd.read_csv(base_url + "patients.csv", dtype={'ZIP': str, 'FIPS': str})
geo.columns = [c.lower() for c in geo.columns]
p = geo

MA_LAT, MA_LON = (41.2, 42.9), (-73.6, -69.9)   # Massachusetts bounding box
MA_COUNTY_FIPS = {'Barnstable':25001,'Berkshire':25003,'Bristol':25005,'Dukes':25007,
    'Essex':25009,'Franklin':25011,'Hampden':25013,'Hampshire':25015,'Middlesex':25017,
    'Nantucket':25019,'Norfolk':25021,'Plymouth':25023,'Suffolk':25025,'Worcester':25027}

def show(title, mask, cols):
    print(f"\n{title}: {int(mask.sum())}")
    if mask.sum():
        display(p.loc[mask, cols])

# 1. Missing values
print("Missing:\n", p[['state','county','city','zip','fips','lat','lon']].isna().sum())

# 2. ZIP format and range
p['zip_s'] = p['zip'].str.strip()
show("ZIP not numeric", ~p['zip_s'].str.fullmatch(r'\d+', na=False), ['id','city','zip'])
p['zip5'] = p['zip_s'].str.zfill(5)
show("ZIP placeholder 00000", p['zip5'] == '00000', ['city','county','zip5'])
valid_zip = p['zip5'] != '00000'
show("ZIP outside MA range (01001-02791)",
     valid_zip & ~p['zip5'].between('01001', '02791'), ['city','zip5'])

# 3. FIPS format
p['fips_s'] = p['fips'].str.replace(r'\.0$', '', regex=True)
show("FIPS not 5 digits", p['fips_s'].notna() & ~p['fips_s'].str.fullmatch(r'\d{5}', na=False),
     ['city','county','fips'])
show("FIPS not in Massachusetts (should start with 25)",
     p['fips_s'].notna() & ~p['fips_s'].str.startswith('25', na=False),
     ['city','county','state','fips_s'])

# 4. County vs FIPS consistency
p['county_name'] = p['county'].str.replace(' County', '', regex=False).str.strip()
p['expected_fips'] = p['county_name'].map(MA_COUNTY_FIPS).astype('Int64').astype(str)
show("FIPS disagrees with county name",
     p['fips_s'].notna() & (p['fips_s'] != p['expected_fips']),
     ['city','county','fips_s','expected_fips'])
print("\nMissing FIPS by county:\n", p[p['fips_s'].isna()]['county'].value_counts())

# 5. State / county / city / ZIP hierarchy
print("\nStates:", p['state'].value_counts().to_dict())
show("County not a MA county", ~p['county_name'].isin(MA_COUNTY_FIPS), ['city','county'])
multi = p.groupby('city')['county'].nunique()
show("City appears in >1 county", p['city'].isin(multi[multi > 1].index), ['city','county'])
multi_z = p[valid_zip].groupby('zip5')['city'].nunique()
show("ZIP maps to >1 city", p['zip5'].isin(multi_z[multi_z > 1].index), ['zip5','city','county'])

# 6. Coordinates
show("Missing lat/lon", p[['lat','lon']].isna().any(axis=1), ['city','lat','lon'])
show("Lat outside [-90, 90]", ~p['lat'].between(-90, 90) & p['lat'].notna(), ['city','lat'])
show("Lon outside [-180, 180]", ~p['lon'].between(-180, 180) & p['lon'].notna(), ['city','lon'])
show("(0,0) or swapped-looking coordinates",
     ((p['lat']==0)&(p['lon']==0)) | (p['lat'].abs() > 60) & (p['lon'].abs() < 60),
     ['city','lat','lon'])
show("Coordinates outside Massachusetts bounding box",
     ~p['lat'].between(*MA_LAT) | ~p['lon'].between(*MA_LON),
     ['city','county','lat','lon'])

# 7. Coordinates vs county (distance from each county's median point)
med = p.groupby('county')[['lat','lon']].transform('median')
p['km_from_county_median'] = np.hypot((p['lat']-med['lat'])*111, (p['lon']-med['lon'])*82)
show("Coordinates >60 km from their county's median point",
     p['km_from_county_median'] > 60, ['city','county','lat','lon','km_from_county_median'])

Missing:
 state      0
county     0
city       0
zip        0
fips      30
lat        0
lon        0
dtype: int64

ZIP not numeric: 0

ZIP placeholder 00000: 30


,city,county,zip5
12,Middleborough,Plymouth County,00000
16,Plympton,Plymouth County,00000
19,Wendell,Franklin County,00000
21,Northbridge,Worcester County,00000
25,Halifax,Plymouth County,00000
28,Plympton,Plymouth County,00000
34,Sterling,Worcester County,00000
35,Plympton,Plymouth County,00000
36,Mansfield,Bristol County,00000
38,Norton,Bristol County,00000



ZIP outside MA range (01001-02791): 1


,city,zip5
75,Attleboro,02861



FIPS not 5 digits: 0

FIPS not in Massachusetts (should start with 25): 1


...


FIPS disagrees with county name: 1


,city,county,fips_s,expected_fips
75,Attleboro,Bristol County,44007,25005



Missing FIPS by county:
 county
Plymouth County     6
Worcester County    6
Bristol County      5
Middlesex County    5
Norfolk County      3
Hampshire County    2
Franklin County     1
Berkshire County    1
Hampden County      1
Name: count, dtype: int64

States: {'Massachusetts': 120}

County not a MA county: 0

City appears in >1 county: 0

ZIP maps to >1 city: 2


,zip5,city,county
15,01906,Lynn,Essex County
108,01906,Saugus,Essex County



Missing lat/lon: 0

Lat outside [-90, 90]: 0

Lon outside [-180, 180]: 0

(0,0) or swapped-looking coordinates: 0

Coordinates outside Massachusetts bounding box: 0

Coordinates >60 km from their county's median point: 0


In [31]:
MA_COUNTY_FIPS = {'Barnstable':'25001','Berkshire':'25003','Bristol':'25005','Dukes':'25007',
    'Essex':'25009','Franklin':'25011','Hampden':'25013','Hampshire':'25015','Middlesex':'25017',
    'Nantucket':'25019','Norfolk':'25021','Plymouth':'25023','Suffolk':'25025','Worcester':'25027'}

# `geo` is the string-typed reload from the geographic check
g = geo.set_index('id')
g['county_name'] = g['county'].str.replace(' County', '', regex=False).str.strip()

# ZIP: keep leading zeros, placeholder -> NULL
g['zip'] = g['zip'].str.zfill(5).replace('00000', np.nan)
# Attleboro row with an RI ZIP -> NULL
g.loc[(g['city'] == 'Attleboro') & (g['zip'] == '02861'), 'zip'] = np.nan
# FIPS: derived from county (fixes the 30 missing and the Attleboro 44007 row)
g['fips'] = g['county_name'].map(MA_COUNTY_FIPS)

pt = data['patients']
pt['zip']  = pt['id'].map(g['zip'])
pt['fips'] = pt['id'].map(g['fips'])

# Readable labels; missing values stay NULL
pt['marital'] = pt['marital'].map({'M': 'married', 'S': 'single', 'D': 'divorced', 'W': 'widowed'})
pt['gender']  = pt['gender'].map({'F': 'female', 'M': 'male'})

print(pt[['zip', 'fips', 'marital', 'gender']].isna().sum())

zip        31
fips        0
marital    39
gender      0
dtype: int64


In [33]:
print(pt['fips'].str.len().value_counts().to_dict(), pt['zip'].str.len().value_counts().to_dict())
print(pt['fips'].dtype, pt['zip'].dtype)

{5: 120} {5.0: 89}
object object


### Patients: geographic fixes and encoding

- ZIP placeholder `00000` (30 rows) and the Attleboro row's RI ZIP `02861` set to NULL.
- FIPS derived from county name: fills 30 missing and fixes the Attleboro row (`44007` → `25005`).
- `zip` and `fips` kept as text to preserve leading zeros.
- `marital` and `gender` codes mapped to readable labels (e.g. `M` → `married`). Missing values stay NULL (mostly minors).

In [34]:
fin = ['healthcare_expenses', 'healthcare_coverage', 'income']
pt = data['patients']

# Age at death, or today for living patients (used to judge whether values are plausible)
end = pt['deathdate'].fillna(pd.Timestamp('2026-09-30'))
pt['age'] = (end - pt['birthdate']).dt.days / 365.25

# 1. Missing, negative, zero
print("Missing:", pt[fin].isna().sum().to_dict())
for c in fin:
    print(f"{c}: negative={(pt[c] < 0).sum()}, zero={(pt[c] == 0).sum()}")
print(pt[fin].describe().T)

# 2. Outliers: IQR x3 fence and log z-score, reported per column (nothing is removed)
flags = pd.DataFrame(index=pt.index)
for c in fin:
    q1, q3 = pt[c].quantile([.25, .75])
    fence = q3 + 3 * (q3 - q1)
    logz = np.log1p(pt[c])
    logz = (logz - logz.mean()) / logz.std()
    flags[f'{c}_iqr'] = pt[c] > fence
    flags[f'{c}_logz'] = logz.abs() > 3
    print(f"{c}: fence={fence:,.0f}, above fence={flags[f'{c}_iqr'].sum()}, "
          f"log-z>3={flags[f'{c}_logz'].sum()}, max={pt[c].max():,.0f}")

# 3. Context for judging legitimacy
print(pt[fin + ['age']].corr(method='spearman').round(2))

pt['expenses_per_year'] = pt['healthcare_expenses'] / pt['age'].clip(lower=1)
cols = ['age', 'deathdate'] + fin + ['expenses_per_year']
print("\nTop expenses:");  display(pt.nlargest(8, 'healthcare_expenses')[cols])
print("\nTop income:");    display(pt.nlargest(6, 'income')[cols])
print("\nTop coverage:");  display(pt.nlargest(6, 'healthcare_coverage')[cols])
print("\nLowest income:"); display(pt.nsmallest(5, 'income')[cols])
print("\nZero coverage:"); display(pt[pt['healthcare_coverage'] == 0][cols])

# 4. Suspicious repeated values
print(pt['income'].value_counts().head(5))

Missing: {'healthcare_expenses': 0, 'healthcare_coverage': 0, 'income': 0}
healthcare_expenses: negative=0, zero=0
healthcare_coverage: negative=0, zero=9
income: negative=0, zero=0
                     count           mean            std    min         25%         50%          75%         max
healthcare_expenses  120.0  152052.529083  191023.194260  400.0  24538.6275   89918.065  204184.6900   991285.28
healthcare_coverage  120.0  348244.126500  431150.652181    0.0  18157.1625  165337.015  584063.1675  1946173.69
income               120.0  120476.175000  174472.938303   71.0  33128.7500   79951.000  124338.2500   981530.00
healthcare_expenses: fence=743,123, above fence=5, log-z>3=1, max=991,285
healthcare_coverage: fence=2,281,781, above fence=0, log-z>3=0, max=1,946,174
income: fence=397,967, above fence=6, log-z>3=1, max=981,530
                     healthcare_expenses  healthcare_coverage  income   age
healthcare_expenses                 1.00                 0.44    0.39  0.68
h

,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
90,61.095140,NaT,991285.28,573630.94,47167,16225.272172
53,62.625599,NaT,938185.96,439743.12,102442,14980.870066
29,71.364819,NaT,856338.00,629643.30,65031,11999.441974
110,81.664613,NaT,821047.68,342159.19,94798,10053.897852
117,64.427105,1986-02-23,773692.06,377262.92,79951,12008.797591
8,61.133470,NaT,591968.83,300378.23,95702,9683.219811
44,49.226557,NaT,506217.48,95602.97,37682,10283.422390
18,59.726215,NaT,487734.33,249038.25,82956,8166.168418



Top income:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
27,42.951403,NaT,180000.31,538083.88,981530,4190.789981
103,48.566735,NaT,139764.65,650058.40,938744,2877.785581
108,46.992471,NaT,99565.36,31761.62,902392,2118.751325
13,52.164271,NaT,35849.94,162744.93,875139,687.250857
91,73.352498,NaT,164964.52,758008.33,634691,2248.928446
70,62.540726,NaT,186148.62,719021.13,472641,2976.438447



Top coverage:


...


Lowest income:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
19,15.178645,NaT,24719.46,21918.32,71,1628.568320
89,3.411362,NaT,400.00,11107.77,2426,117.255217
39,67.457906,NaT,18474.70,974308.91,4412,273.870051
26,34.636550,NaT,11706.74,291852.88,4582,337.988047
12,64.016427,NaT,19115.40,1417903.14,7029,298.601482



Zero coverage:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
5,5.486653,NaT,8837.38,0.0,50296,1610.705112
17,39.455168,NaT,135793.50,0.0,101170,3441.716458
23,6.340862,NaT,14571.95,0.0,53058,2298.102218
25,1.141684,NaT,3998.87,0.0,51090,3502.607356
38,19.195072,2026-06-18,110717.12,0.0,70638,5767.997159
40,1.018480,NaT,4926.41,0.0,98733,4837.019496
45,40.161533,2024-10-12,174083.26,0.0,76288,4334.577048
50,42.127310,NaT,109858.19,0.0,76288,2607.766550
102,11.613963,NaT,28641.53,0.0,47790,2466.128909


income
79951     8
24699     4
82756     3
82956     2
250398    2
Name: count, dtype: int64


### Checked for incomes, expences - nothing needs to be changed

In [28]:
# Configure pandas to display all columns for each table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

for name, df in data.items():
    print(f"\n{'='*50}\nTable: {name.upper()} {df.shape}\n{'='*50}")
    display(df.head(5))
    print("\n")
    df.info()




Table: PATIENTS (120, 29)


,id,birthdate,deathdate,ssn,drivers,passport,prefix,first,middle,last,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,fips,zip,lat,lon,healthcare_expenses,healthcare_coverage,income,age_at_reference
0,f19b9cae-0ea7-2d51-f2c5-946d448992a5,2020-01-15,NaT,999-65-3365,NaN,NaN,NaN,Erick204,Nicolas769,Stark857,NaN,NaN,NaN,white,nonhispanic,M,Chelmsford Massachusetts US,966 Ortiz Park,Quincy,Massachusetts,Norfolk County,25021.0,2171,42.273142,-71.053109,4047.16,5278.91,31892,6.959617
1,4cdd696b-7a6c-dffe-b1ba-66b561ddf55d,1993-10-10,NaT,999-92-4099,S99922748,X87859644X,Mr.,Elwood28,Dario75,Klocko335,NaN,NaN,S,white,nonhispanic,M,Brookline Massachusetts US,1087 Walter Frontage road,Amesbury,Massachusetts,Essex County,25009.0,1913,42.825606,-70.956765,49458.46,35754.23,85561,33.223819
2,2230008d-767a-e6bb-a12f-47361158ae16,1980-02-29,NaT,999-23-3484,S99931870,X55853975X,Mrs.,Scarlett814,Dyan186,Cronin387,NaN,Little434,W,white,nonhispanic,F,Revere Massachusetts US,206 Green Vale,Braintree,Massachusetts,Norfolk County,25021.0,2184,42.246404,-70.972495,167999.42,309913.70,131899,46.836413
3,e18a2f81-3870-eabf-dd21-8d9bcf87361b,2015-04-18,NaT,999-42-9123,NaN,NaN,NaN,Gregory545,Eliseo499,Greenholt190,NaN,NaN,NaN,white,nonhispanic,M,Quincy Massachusetts US,291 Krajcik Spur,New Bedford,Massachusetts,Bristol County,25005.0,2740,41.602433,-70.919064,2602.58,57929.92,18491,11.704312
4,cbd98cb4-5a3a-1c20-7d4c-5842206c3b3f,1977-11-04,NaT,999-13-7294,S99987596,X47556351X,Mrs.,Theresa98,Vania595,Ferry570,NaN,Champlin946,M,white,nonhispanic,F,North Attleborough Massachusetts US,313 Bosco Bridge,Pepperell,Massachusetts,Middlesex County,25017.0,1463,42.714593,-71.607422,340943.34,197539.35,66229,49.155373




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   120 non-null    object        
 1   birthdate            120 non-null    datetime64[ns]
 2   deathdate            20 non-null     datetime64[ns]
 3   ssn                  120 non-null    object        
 4   drivers              98 non-null     object        
 5   passport             89 non-null     object        
 6   prefix               94 non-null     object        
 7   first                120 non-null    object        
 8   middle               89 non-null     object        
 9   last                 120 non-null    object        
 10  suffix               1 non-null      object        
 11  maiden               37 non-null     object        
 12  marital              81 non-null     object        
 13  race                 120 non-null

,id,start,stop,patient,organization,provider,payer,encounterclass,code,description,base_encounter_cost,total_claim_cost,payer_coverage,reasoncode,reasondescription,is_death_certification
0,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,2020-01-14T22:59:24Z,2020-01-14T23:14:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,39c15c0f-5c49-311e-99d2-1fb99d80e06e,d8c55878-c9f6-3e9f-9cf0-a2f844c211ad,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,347.38,247.38,NaN,NaN,False
1,4cdd696b-7a6c-dffe-1831-f25b8aaa4792,2005-04-14T22:53:06Z,2005-04-14T23:08:06Z,4cdd696b-7a6c-dffe-b1ba-66b561ddf55d,18a96051-91f4-3cba-af96-056ea14dfee0,7bb499fe-443e-3c65-9e25-92ea93b228f1,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185345009,Encounter for symptom (procedure),85.55,92.93,0.00,36971009.0,Sinusitis (disorder),False
2,2230008d-767a-e6bb-5cb9-2a8772770479,1997-04-18T14:44:54Z,1997-04-18T15:10:04Z,2230008d-767a-e6bb-a12f-47361158ae16,69eae27b-7445-3652-bd34-b535abe0b011,ece20e64-0974-37db-907c-84deb926474d,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,wellness,410620009,Well child visit (procedure),136.80,1416.25,0.00,NaN,NaN,False
3,2230008d-767a-e6bb-d253-c5d894fa1803,1998-04-24T14:44:54Z,1998-04-24T15:29:20Z,2230008d-767a-e6bb-a12f-47361158ae16,69eae27b-7445-3652-bd34-b535abe0b011,ece20e64-0974-37db-907c-84deb926474d,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,wellness,162673000,General examination of patient (procedure),136.80,919.90,0.00,NaN,NaN,False
4,2230008d-767a-e6bb-fd67-8d0bffdaed48,2002-05-03T14:44:54Z,2002-05-03T15:33:18Z,2230008d-767a-e6bb-a12f-47361158ae16,69eae27b-7445-3652-bd34-b535abe0b011,ece20e64-0974-37db-907c-84deb926474d,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,wellness,162673000,General examination of patient (procedure),136.80,1642.18,0.00,NaN,NaN,False




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7042 entries, 0 to 7041
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      7042 non-null   object 
 1   start                   7042 non-null   object 
 2   stop                    7042 non-null   object 
 3   patient                 7042 non-null   object 
 4   organization            7042 non-null   object 
 5   provider                7042 non-null   object 
 6   payer                   7042 non-null   object 
 7   encounterclass          7042 non-null   object 
 8   code                    7042 non-null   int64  
 9   description             7042 non-null   object 
 10  base_encounter_cost     7042 non-null   float64
 11  total_claim_cost        7042 non-null   float64
 12  payer_coverage          7042 non-null   float64
 13  reasoncode              3833 non-null   float64
 14  reasondescription       3833 non-null 

...



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8001 entries, 0 to 8000
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   start              8001 non-null   object 
 1   stop               7653 non-null   object 
 2   patient            8001 non-null   object 
 3   payer              8001 non-null   object 
 4   encounter          8001 non-null   object 
 5   code               8001 non-null   int64  
 6   description        8001 non-null   object 
 7   base_cost          8001 non-null   float64
 8   payer_coverage     8001 non-null   float64
 9   dispenses          8001 non-null   int64  
 10  totalcost          8001 non-null   float64
 11  reasoncode         6954 non-null   float64
 12  reasondescription  6954 non-null   object 
dtypes: float64(4), int64(2), object(7)
memory usage: 812.7+ KB

Table: OBSERVATIONS (117502, 10)


,date,patient,encounter,category,code,description,value,units,type,is_cause_of_death
0,2020-01-14T22:59:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,vital-signs,8302-2,Body Height,51.5,cm,numeric,False
1,2020-01-14T22:59:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,1.0,{score},numeric,False
2,2020-01-14T22:59:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,vital-signs,29463-7,Body Weight,3.8,kg,numeric,False
3,2020-01-14T22:59:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,vital-signs,77606-2,Weight-for-length Per age and sex,33.9,%,numeric,False
4,2020-01-14T22:59:24Z,f19b9cae-0ea7-2d51-f2c5-946d448992a5,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,vital-signs,8289-1,Head Occipital-frontal circumference Percentile,34.9,%,numeric,False




<class 'pandas.core.frame.DataFrame'>
Index: 117502 entries, 0 to 117523
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   date               117502 non-null  object
 1   patient            117502 non-null  object
 2   encounter          114313 non-null  object
 3   category           114313 non-null  object
 4   code               117502 non-null  object
 5   description        117502 non-null  object
 6   value              117502 non-null  object
 7   units              85654 non-null   object
 8   type               117502 non-null  object
 9   is_cause_of_death  117502 non-null  bool  
dtypes: bool(1), object(9)
memory usage: 9.1+ MB

Table: PROCEDURES (21901, 10)


,start,stop,patient,encounter,system,code,description,base_cost,reasoncode,reasondescription
0,2018-11-01T16:53:06Z,2018-11-01T17:11:45Z,4cdd696b-7a6c-dffe-b1ba-66b561ddf55d,4cdd696b-7a6c-dffe-49b1-f88d17db4900,http://snomed.info/sct,23426006,Measurement of respiratory function (procedure),440.55,10509002.0,Acute bronchitis (disorder)
1,2017-03-25T01:19:31Z,2017-03-25T01:34:31Z,e18a2f81-3870-eabf-dd21-8d9bcf87361b,e18a2f81-3870-eabf-5554-c5f56e4e8ea3,http://snomed.info/sct,430193006,Medication reconciliation (procedure),244.56,NaN,NaN
2,2017-05-19T14:44:54Z,2017-05-19T14:59:54Z,2230008d-767a-e6bb-a12f-47361158ae16,2230008d-767a-e6bb-1768-e55eb6249eaf,http://snomed.info/sct,430193006,Medication reconciliation (procedure),271.65,NaN,NaN
3,2017-05-19T14:44:54Z,2017-05-19T15:43:02Z,2230008d-767a-e6bb-a12f-47361158ae16,2230008d-767a-e6bb-1768-e55eb6249eaf,http://snomed.info/sct,710824005,Assessment of health and social care needs (pr...,431.40,NaN,NaN
4,2017-05-19T15:43:02Z,2017-05-19T16:04:25Z,2230008d-767a-e6bb-a12f-47361158ae16,2230008d-767a-e6bb-1768-e55eb6249eaf,http://snomed.info/sct,710841007,Assessment of anxiety (procedure),431.40,NaN,NaN




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21901 entries, 0 to 21900
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   start              21901 non-null  object 
 1   stop               21901 non-null  object 
 2   patient            21901 non-null  object 
 3   encounter          21901 non-null  object 
 4   system             21901 non-null  object 
 5   code               21901 non-null  int64  
 6   description        21901 non-null  object 
 7   base_cost          21901 non-null  float64
 8   reasoncode         9373 non-null   float64
 9   reasondescription  9373 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 1.7+ MB
